# Week 8 — Ensemble Methods: Random Forest
## Combining many trees into one prediction

An **ensemble** combines several models so that their individual mistakes partly cancel out. A **Random Forest** is one of the most widely used ensembles. It builds many decision trees and lets them vote:

1. **Bootstrap samples:** each tree is trained on its own sample drawn from the training data *with replacement*, so every tree sees a slightly different dataset.
2. **Random feature subsets:** at every split, a tree may only choose from a random subset of the features. This stops all trees from relying on the same strong feature and makes their errors less alike.
3. **Voting:** for a new observation, every tree makes a prediction and the forest combines them (majority vote for classification, average for regression).

A single deep tree is unstable: small changes in the data can produce a very different tree. Averaging many different trees reduces this variance.

In this notebook we:

- train a Random Forest
- look at how individual trees disagree and how the forest combines them
- evaluate the forest with cross-validation
- see how the number of trees affects performance
- inspect which features the forest relies on
- evaluate the final model once on a held-out test set

We again use a simulated binary classification dataset in data folder for teaching. It is artificial and does not describe real students or real-world outcomes.

## 1. Prepare the data

The dataset has five simulated indicators and a binary outcome, `needs_support`. We set aside 20% of the rows as a test set and do not touch them until the final evaluation. All other evaluation uses cross-validation on the training set.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42  # fixes the randomness

feature_names = ["attendance", "study_time", "engagement", "assessment_score", "help_seeking"]

df = pd.read_csv("../data/week8_evaluation_synt_data.csv")
X = df[feature_names]
y = df["needs_support"]

# stratify=y keeps the same class proportions in the training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train), "| Test rows (kept aside):", len(X_test))
y_train.value_counts(normalize=True).rename("training class proportion")

About 29% of training rows belong to class 1 (`needs_support`). Because the classes are imbalanced, accuracy alone can look good even if the model misses many class 1 cases, so we will also look at precision, recall and F1.

## 2. Train a Random Forest

The main settings:

- `n_estimators`: how many trees to build.
- `max_features`: how many features each split may choose from. `"sqrt"` means √5 ≈ 2 of the 5 features, picked at random for every split.
- `bootstrap=True`: each tree gets its own sample drawn with replacement (the default).
- `random_state`: makes the random sampling reproducible.

The trees are grown fully (no depth limit). Each tree on its own would overfit, but averaging many of them reduces this.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=200,       # number of trees
    max_features="sqrt",    # random subset of features considered at each split
    bootstrap=True,         # each tree trains on a bootstrap sample (default)
    random_state=RANDOM_STATE,
    n_jobs=-1               # train trees in parallel on all CPU cores
)
forest.fit(X_train, y_train)
print("Number of trees in the forest:", len(forest.estimators_))

## 3. Inside the forest: trees disagree, the forest votes

Every tree saw a different bootstrap sample and different feature subsets, so the trees do not always agree. The table shows training rows where the first five trees disagree, next to the forest's final prediction.

Scikit-learn's forest averages the trees' predicted class probabilities. With fully grown trees each tree predicts 0 or 1 with certainty, so this is the same as a majority vote.

In [ ]:
# Predictions of the first five trees (the trees were fitted on plain arrays, hence .to_numpy())
votes = pd.DataFrame(
    {f"tree {i}": tree.predict(X_train.to_numpy()).astype(int)
     for i, tree in enumerate(forest.estimators_[:5], start=1)},
    index=X_train.index
)
votes["forest (all 200 trees)"] = forest.predict(X_train)

# Keep only the rows where the five trees do not all agree
votes[votes.iloc[:, :5].nunique(axis=1) > 1].head()

## 4. Evaluate with cross-validation

A model's score on its own training data is not a fair measure: a forest of fully grown trees can fit the training data almost perfectly. Instead we use **5-fold cross-validation**: the training set is split into 5 parts, the forest is trained on 4 and scored on the 5th, and this is repeated so every part is held out once.

We report the mean and the standard deviation across the 5 folds. The standard deviation shows how much the score depends on which rows end up in which fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # same class mix in every fold
metrics = ["accuracy", "precision", "recall", "f1"]

scores = cross_validate(forest, X_train, y_train, cv=cv, scoring=metrics)

cv_summary = pd.DataFrame({
    "mean": [scores[f"test_{m}"].mean() for m in metrics],
    "SD":   [scores[f"test_{m}"].std() for m in metrics],
}, index=metrics)
cv_summary.round(3)

## 5. How many trees are enough?

More trees make the vote more stable, but the gain levels off while the training time keeps growing. Below we repeat the cross-validation for forests of different sizes. Once the curve flattens, small ups and downs between sizes are noise from the folds, not real differences.

In [ ]:
tree_counts = [1, 5, 10, 50, 100, 200]
f1_by_size = []

for n in tree_counts:
    model = RandomForestClassifier(n_estimators=n, max_features="sqrt",
                                   random_state=RANDOM_STATE, n_jobs=-1)
    f1 = cross_validate(model, X_train, y_train, cv=cv, scoring="f1")["test_score"]
    f1_by_size.append(f1.mean())

plt.plot(tree_counts, f1_by_size, marker="o")
plt.xscale("log")
plt.xticks(tree_counts, tree_counts)
plt.xlabel("Number of trees (log scale)")
plt.ylabel("Mean cross-validated F1")
plt.title("Adding trees helps at first, then levels off")
plt.show()

## 6. Which features does the forest rely on?

A forest of 200 trees is much harder to read than a single tree. **Feature importance** gives a summary: how much each feature reduced impurity across all splits in all trees, scaled to sum to 1.

These importances describe what the model uses, not what causes the outcome. They can also be misleading when features are strongly correlated, because the importance is split between them.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=feature_names).sort_values()

importances.plot.barh()
plt.xlabel("Importance (sums to 1)")
plt.title("Random Forest feature importances")
plt.show()

## 7. Final evaluation on the test set

Now we use the test set, once. The forest was already trained on the full training set in section 2.

In [ ]:
test_pred = forest.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap="Blues")
plt.title("Random Forest: test set confusion matrix")
plt.show()

### Questions

1. In cross-validation, recall for class 1 is lower than precision. In a setting where the goal is to identify students who need support, which error is more costly: missing a student, or flagging one who does not need help?
2. What happens if you set `max_features=None`, so every split can use all features? (That forest is plain bagging.)
3. From the plot in section 5, how many trees would you choose, and why?
4. Why is a Random Forest less interpretable than a single decision tree?

## 8. Key takeaways

- A **Random Forest** trains many decision trees, each on a **bootstrap sample** and with a **random subset of features at each split**, and combines them by voting.
- The randomness makes the trees disagree; averaging their predictions mainly **reduces variance** compared with a single deep tree.
- **Feature importances** summarise what the forest relies on, but they do not show causation.
- Use **cross-validation** to evaluate and compare settings, and keep the **test set** for one final check.